# Prototype 2: RoPE + Multi-Layer Transformer

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time

## 1. RoPE

In [2]:
class RoPE(nn.Module):
    def __init__(self, dim, max_len=1000000, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
    def forward(self, x, offset=0):
        seq_len = x.shape[-2]
        t = torch.arange(offset, offset + seq_len, device=x.device).type_as(self.inv_freq)
        freqs = t.unsqueeze(-1) @ self.inv_freq.unsqueeze(0)
        return torch.cat([freqs, freqs], dim=-1)

def apply_rope(x, emb):
    s = x.shape[-1] // 2
    xc = torch.view_as_complex(x.reshape(*x.shape[:-1], s, 2).contiguous())
    ec = torch.view_as_complex(emb.reshape(*emb.shape[:-1], s, 2).contiguous())
    return torch.view_as_real(xc * ec).reshape(*x.shape[:-1], -1)

## 2. Core Layers

In [3]:
class LinearAttn(nn.Module):
    def __init__(self, dim, decay=0.99, eps=1e-6):
        super().__init__(); self.decay, self.eps = decay, eps
    def fm(self, x): return F.elu(x) + 1.0
    def forward(self, q, k, v, S=None, z=None):
        qf, kf, vf = self.fm(q), self.fm(k), self.fm(v)
        if S is None:
            S = kf.new_zeros(*q.shape[:2], q.shape[-1], q.shape[-1])
            z = kf.new_zeros(*q.shape[:2], q.shape[-1])
        S = self.decay * S + kf.transpose(-2, -1) @ vf
        z = self.decay * z + kf.sum(-2)
        return (qf @ S) / (qf @ z.unsqueeze(-1)).clamp(min=self.eps), S, z

class KVCompress(nn.Module):
    def __init__(self, dim, hd, m, overlap=False):
        super().__init__()
        self.m, self.overlap = m, overlap
        self.W_k = nn.Linear(dim, hd, 0); self.W_v = nn.Linear(dim, hd, 0)
        self.W_z = nn.Linear(dim, 1, 0)
        if overlap:
            self.W_kb = nn.Linear(dim, hd, 0); self.W_vb = nn.Linear(dim, hd, 0)
            self.W_zb = nn.Linear(dim, 1, 0)
    def forward(self, h):
        b, n, d = h.shape; m = self.m; nc = n // m; r = nc * m
        hb = h[:, :r].reshape(b, nc, m, d)
        k = self.W_k(hb); v = self.W_v(hb); z = self.W_z(hb).squeeze(-1)
        if self.overlap and nc > 1:
            hs = torch.cat([h[:, :1].expand(-1, m, -1), h[:, :r]], 1)[:, :r]
            hs = hs.reshape(b, nc, m, d)
            kb = self.W_kb(hs); vb = self.W_vb(hs); zb = self.W_zb(hs).squeeze(-1)
            k = torch.cat([k, kb], 2); v = torch.cat([v, vb], 2)
            z = torch.cat([z, zb], -1)
        w = F.softmax(z, -1)
        return (w.unsqueeze(-1) * k).sum(2), (w.unsqueeze(-1) * v).sum(2)

## 3. Attention Layers

In [4]:
class CSA(nn.Module):
    def __init__(self, dim, hd, m=4, tk=64, decay=0.99):
        super().__init__()
        self.tk = tk; self.cp = KVCompress(dim, hd, m, True)
        self.qp = nn.Linear(dim, hd, 0); self.out = nn.Linear(hd, dim, 0)
        self.la = LinearAttn(hd, decay)
    def forward(self, q, h, S=None, z_s=None):
        k, v = self.cp(h); qk = self.qp(q[:, -1:])
        s = (qk @ k.transpose(-2, -1)) / math.sqrt(k.shape[-1])
        _, ix = torch.topk(s, min(self.tk, k.shape[1]), -1)
        ix = ix.unsqueeze(-1).expand(-1, -1, -1, k.shape[-1]).squeeze(1)
        k, v = torch.gather(k, 1, ix), torch.gather(v, 1, ix)
        o, S, z_s = self.la(qk.unsqueeze(1), k.unsqueeze(1), v.unsqueeze(1), S, z_s)
        return self.out(o.squeeze(1)), S, z_s

class HCA(nn.Module):
    def __init__(self, dim, hd, m=128, decay=0.999):
        super().__init__()
        self.cp = KVCompress(dim, hd, m, False); self.out = nn.Linear(hd, dim, 0)
        self.qp = nn.Linear(dim, hd, 0); self.la = LinearAttn(hd, decay)
    def forward(self, q, h, S=None, z_s=None):
        k, v = self.cp(h); qk = self.qp(q[:, -1:]).unsqueeze(1)
        o, S, z_s = self.la(qk, k.unsqueeze(1), v.unsqueeze(1), S, z_s)
        return self.out(o.squeeze(1)), S, z_s

class SWA(nn.Module):
    def __init__(self, dim, nh=4, win=128):
        super().__init__(); self.win=win; self.nh=nh; self.hd=dim//nh
        self.qp=nn.Linear(dim,dim,0); self.kp=nn.Linear(dim,dim,0)
        self.vp=nn.Linear(dim,dim,0); self.op=nn.Linear(dim,dim,0)
        self.rope = RoPE(dim // nh)
    def forward(self, q, h, offset=0):
        b,n,d=h.shape; c=h[:,-min(n,self.win):]
        k=self.kp(c).view(b,-1,self.nh,self.hd)
        v=self.vp(c).view(b,-1,self.nh,self.hd)
        qk=self.qp(q[:,-1:]).view(b,1,self.nh,self.hd)
        emb=self.rope(k,offset); k=apply_rope(k,emb)
        emb_q=self.rope(qk,offset+k.shape[1]-1); qk=apply_rope(qk,emb_q)
        k=k.transpose(1,2); v=v.transpose(1,2); qk=qk.transpose(1,2)
        a=F.softmax((qk@k.transpose(-2,-1))/math.sqrt(self.hd),-1)
        return self.op((a@v).transpose(1,2).reshape(b,1,d).squeeze(1))

## 4. Multi-Layer Block

In [5]:
class HybridLayer(nn.Module):
    def __init__(self, dim=256, hd=64, nh=4, cm=4, ck=64, cd=0.99, hm=128, hd2=0.999, sw=128):
        super().__init__()
        self.csa = CSA(dim, hd, cm, ck, cd)
        self.hca = HCA(dim, hd, hm, hd2)
        self.swa = SWA(dim, nh, sw)
        self.g = nn.Parameter(torch.ones(3))
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, q, h, cs=None, cz=None, hs=None, hz=None, offset=0):
        co,cs,cz=self.csa(q,h,cs,cz); ho,hs,hz=self.hca(q,h,hs,hz)
        so=self.swa(q,h,offset); g=F.softmax(self.g,0)
        out=self.ffn(self.norm(q[:,-1:]+g[0]*co+g[1]*ho+g[2]*so))
        return out+q[:,-1:], (cs,cz), (hs,hz)

## 5. Full Transformer

In [6]:
class EdgeTransformer(nn.Module):
    def __init__(self, n_layers=4, dim=256, hd=64, nh=4, vocab_size=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([HybridLayer(dim, hd, nh) for _ in range(n_layers)])
        self.lm_head = nn.Linear(dim, vocab_size, 0)
    def forward(self, tokens):
        h = self.embed(tokens)
        cs=[None]*len(self.layers); cz=[None]*len(self.layers)
        hs=[None]*len(self.layers); hz=[None]*len(self.layers)
        for i, l in enumerate(self.layers):
            h,(cs[i],cz[i]),(hs[i],hz[i]) = l(h, h, cs[i], cz[i], hs[i], hz[i])
        return self.lm_head(h[:, -1:])

## 6. Test

In [7]:
m = EdgeTransformer(n_layers=2, dim=128, hd=32, nh=4, vocab_size=100)
x = torch.randint(0, 100, (1, 64))
logits = m(x)
print(f"Output: {list(logits.shape)}")
print(f"Layers: {len(m.layers)}")
print(f"Params: {sum(p.numel() for p in m.parameters()):,}")

Output: [1, 1, 100]
Layers: 2
Params: 503,302


## 7. Stream Test

In [8]:
def stream(m, nf=500, dim=128):
    cs=[None]*len(m.layers); cz=[None]*len(m.layers)
    hs=[None]*len(m.layers); hz=[None]*len(m.layers)
    tok = torch.zeros(1, 1, dtype=torch.long)
    for step in range(0, nf, 128):
        h = m.embed(tok).expand(-1, 128, -1)
        for i, l in enumerate(m.layers):
            h,(cs[i],cz[i]),(hs[i],hz[i]) = l(h, h, cs[i], cz[i], hs[i], hz[i], offset=step)
        tok = m.lm_head(h[:,-1:]).argmax(-1)
    kv = sum(s.element_size()*s.numel() for s in cs+hs)/1024
    print(f"Streamed {nf} frames, KV cache: {kv:.1f} KB ({kv/len(m.layers):.1f} KB/layer)")
    print(f"MHA equiv: {nf*128*4*2/1024**2*len(m.layers):.1f} MB")

stream(m, 500)

Streamed 500 frames, KV cache: 16.0 KB (8.0 KB/layer)
MHA equiv: 1.0 MB
